Общие импорты

In [1]:
import csv
import datetime
import json
from copy import deepcopy
from pathlib import Path
from typing import Any

In [2]:
class Operation:
    _types_operation = {
        "deposit": "deposit",
        "withdraw": "withdraw",
        "interest": "interest",
    }
    _statuses_operation = {
        "success": "success",
        "fail": "fail",
    }

    @staticmethod
    def get_types_operation() -> dict[str, str]:
        return deepcopy(Operation._types_operation)

    @staticmethod
    def get_statuses_operation() -> dict[str, str]:
        return deepcopy(Operation._statuses_operation)

    def __init__(
        self,
        account_number: str,
        type_operation: str,
        statuses_operation: str,
        amount: float,
        current_balance: float,
        *,
        date: datetime.datetime | None = None,
    ) -> None:
        if type_operation not in Operation._types_operation:
            raise ValueError(
                f"Type operation must be either: {Operation._types_operation}"
            )
        if statuses_operation not in Operation._statuses_operation:
            raise ValueError(
                "Status operation must be either: "
                f"{Operation._statuses_operation}"
            )

        self._account_number: str = account_number
        self._type_operation: str = Operation._types_operation[type_operation]
        self._statuses_operation: str = (
            Operation._statuses_operation[statuses_operation]
        )
        self._amount: float = amount
        self._current_balance: float = current_balance
        self._date: datetime.datetime = (
            date if date is not None else datetime.datetime.now()
        )

    @property
    def type_operation(self) -> str:
        return self._type_operation

    @property
    def status(self) -> str:
        return self._statuses_operation

    @property
    def current_balance(self) -> float:
        return self._current_balance

    @property
    def amount(self) -> float:
        return self._amount

    @property
    def date(self) -> datetime.datetime:
        return self._date

    def __str__(self) -> str:
        return (
            f"{self._account_number} "
            f"{self._type_operation} "
            f"{self._statuses_operation} "
            f"{self._amount} "
            f"{self._current_balance} "
            f"{self._date:%Y-%m-%d %H:%M:%S}"
        )

    def __repr__(self) -> str:
        return f"{self.__str__()}"


class Account:
    _account_counter: int = 1000
    account_type: str | None = None
    allowed_operations: frozenset[str] = frozenset({"deposit", "withdraw"})

    def __init__(self, account_holder: str, balance: float = 0) -> None:
        if not self.validate_account_holder(account_holder):
            raise ValueError(
                "The account holder's name must be in the format "
                '"First Name Last Name".'
            )
        if balance < 0:
            raise ValueError("Balance must be non-negative")
        self.holder: str = account_holder
        self.account_number: str = f"ACC-{self._account_counter}"
        self._balance: float = balance
        self._initial_balance: float = balance
        self.operations_history: list[Operation] = []
        self._imported_operations: set[
            tuple[datetime.datetime, str, str, float, float]
        ] = set()

        Account._account_counter += 1

    def log(
        self,
        type_operation: str,
        status: str,
        amount: float,
        current_balance: float,
    ) -> None:
        new_operation = Operation(
            self.account_number,
            type_operation,
            status,
            amount,
            current_balance,
        )
        self.operations_history.append(new_operation)

    def deposit(self, amount: float) -> None:
        if amount <= 0:
            raise ValueError("Amount must be positive.")

        self._balance += amount
        self.log("deposit", "success", amount, self._balance)

    def withdraw(self, amount: float) -> None:
        if amount <= 0:
            raise ValueError("Amount must be positive.")

        if amount > self._balance:
            self.log("withdraw", "fail", amount, self._balance)
            raise ValueError("Insufficient funds.")
        else:
            self._balance -= amount
            self.log("withdraw", "success", amount, self._balance)

    def get_transaction_analysis(
        self, limit: int, min_amount: float = 0
    ) -> list[Operation]:
        if limit < 0:
            raise ValueError("Limit must be non-negative.")
        if min_amount < 0:
            raise ValueError("Minimum amount must be non-negative.")

        operations: list[Operation] = sorted(
            (
                operation
                for operation in reversed(self.get_history())
                if operation.amount >= min_amount
            ),
            key=lambda operation: operation.date,
            reverse=True,
        )
        return operations[:limit]

    @staticmethod
    def validate_account_holder(account_holder: str) -> bool:
        upper: str = (
            "АБВГДЕЁЖЗИЙКЛМНОПРСТУФХЦЧШЩЪЫЬЭЮЯ"
            "ABCDEFGHIJKLMNOPQRSTUVWXYZ"
        )
        letters: str = upper + upper.lower()
        parts: list[str] = account_holder.split(" ")

        return len(parts) == 2 and all(
            part.istitle() and all(char in letters for char in part)
            for part in parts
        )

    def import_history(self, filename: str | Path) -> None:
        path: Path = Path(filename)
        rows: list[dict[str, Any]]
        if path.suffix.lower() == ".csv":
            rows = self.read_csv(path)
        elif path.suffix.lower() == ".json":
            rows = self.read_json(path)
        else:
            raise ValueError("Supported file formats: .csv and .json")

        new_operations: list[Operation] = []
        imported_keys = self._imported_operations.copy()
        for operation in self.clean_history(rows):
            key = (
                operation.date,
                operation.type_operation,
                operation.status,
                operation.amount,
                operation.current_balance,
            )
            if key not in imported_keys:
                imported_keys.add(key)
                new_operations.append(operation)
        if not new_operations:
            return

        operations: list[Operation] = sorted(
            self.operations_history + new_operations,
            key=lambda operation: operation.date,
        )
        balance: float = self._initial_balance
        history: list[Operation] = []
        for operation in operations:
            if operation.status == "success":
                if operation.type_operation == "withdraw":
                    balance -= operation.amount
                else:
                    balance += operation.amount
            if balance < 0:
                raise ValueError(
                    "Imported history produces a negative balance; "
                    "check the initial balance."
                )
            history.append(
                Operation(
                    self.account_number,
                    operation.type_operation,
                    operation.status,
                    operation.amount,
                    balance,
                    date=operation.date,
                )
            )

        self.operations_history = history
        self._balance = balance
        self._imported_operations = imported_keys

    def clean_history(self, rows: list[dict[str, Any]]) -> list[Operation]:
        operations: list[Operation] = []
        statuses: dict[str, str] = Operation.get_statuses_operation()
        for row in rows:
            if row.get("account_number") != self.account_number:
                continue
            if row.get("account_type") not in ("checking", "savings"):
                continue
            if (
                self.account_type is not None
                and row["account_type"] != self.account_type
            ):
                continue
            try:
                type_operation: str = row["operation"]
                status: str = row["status"]
                if (
                    type_operation not in self.allowed_operations
                    or status not in statuses
                ):
                    continue
                if isinstance(row["amount"], bool) or isinstance(
                    row["balance_after"], bool
                ):
                    continue
                amount: float = float(row["amount"])
                balance_after: float = float(row["balance_after"])
                date: datetime.datetime = datetime.datetime.strptime(
                    row["date"], "%Y-%m-%d %H:%M:%S"
                )
                if amount < 0:
                    continue
                if amount == 0 and type_operation != "interest":
                    continue
                if balance_after < 0:
                    continue
            except (KeyError, TypeError, ValueError, OverflowError):
                continue
            operations.append(
                Operation(
                    self.account_number,
                    type_operation,
                    status,
                    amount,
                    balance_after,
                    date=date,
                )
            )
        return operations

    @staticmethod
    def read_csv(filename: str | Path) -> list[dict[str, Any]]:
        with open(filename, "r", encoding="utf-8-sig", newline="") as file:
            return list(csv.DictReader(file))

    @staticmethod
    def read_json(filename: str | Path) -> list[dict[str, Any]]:
        with open(filename, "r", encoding="utf-8-sig") as file:
            rows: Any = json.load(file)
        if not isinstance(rows, list):
            raise ValueError(
                "JSON must contain a list of transaction objects."
            )
        return [row for row in rows if isinstance(row, dict)]

    def get_balance(self) -> float:
        return self._balance

    def get_history(self) -> list[Operation]:
        return deepcopy(self.operations_history)

    def __str__(self) -> str:
        return f"{self.holder} {self.account_number} {self.get_balance()}"

    def __repr__(self) -> str:
        return f"{self.__str__()}"

In [ ]:
class CheckingAccount(Account):
    account_type: str = "checking"

    def __init__(self, account_holder: str, balance: float = 0) -> None:
        super().__init__(account_holder, balance)

In [3]:
class SavingsAccount(Account):
    account_type: str = "savings"
    allowed_operations: frozenset[str] = frozenset(
        {"deposit", "withdraw", "interest"}
    )
    balance_threshold: int = 50

    def __init__(self, account_holder: str, balance: float = 0) -> None:
        super().__init__(account_holder, balance)

    def withdraw(self, amount: float) -> None:
        limit: float = self._balance * (self.balance_threshold / 100)
        if limit < amount <= self._balance:
            self.log("withdraw", "fail", amount, self._balance)
            raise ValueError(
                f"You cannot withdraw more than {self.balance_threshold}% "
                "of the balance."
            )

        super().withdraw(amount)

    def apply_interest(self, rate: float) -> None:
        if rate < 0:
            raise ValueError("Rate must be non-negative.")

        interest_amount: float = self._balance * (rate / 100)
        new_balance: float = self._balance + interest_amount

        self.log("interest", "success", interest_amount, new_balance)
        self._balance = new_balance